# Lab 4 - Classification : prédire une catégorie

**Objectifs du Lab**

À la fin de ce Lab, l'étudiant sera capable de :

- **Entraîner** une régression logistique, un KNN et un arbre de décision sur une même tâche de classification ;
- **Calculer et interpréter** accuracy, précision, rappel, F1-score, ROC-AUC ;
- **Tracer et lire** une courbe ROC et une courbe précision-rappel ;
- **Ajuster le seuil de décision** d'un modèle et en observer l'effet sur précision et rappel ;
- **Gérer** un déséquilibre de classes avec `class_weight` et `stratify` ;
- **Réutiliser** `GridSearchCV` pour régler l'hyperparamètre d'un modèle de classification.

**Téléchargement de la Lab Sheet**

Téléchargez un seul format avant de travailler :

- **Notebook `.ipynb`** pour Google Colab, JupyterLab, Jupyter Notebook ou VS Code : ouvrez le dossier GitHub [`labsheets/`](https://github.com/iaousse/ml-course/tree/main/labsheets) et téléchargez le [fichier source du Lab 4](https://github.com/iaousse/ml-course/blob/main/labsheets/lab4_classification.ipynb).
- **Fichier `.qmd`** pour travailler avec Quarto : ouvrez le dossier GitHub [`tp/`](https://github.com/iaousse/ml-course/tree/main/tp) et téléchargez le [fichier source du Lab 4](https://github.com/iaousse/ml-course/blob/main/tp/lab4_classification.qmd).

## 1. Construire un jeu de données de *churn*

In [ ]:
import numpy as np
import pandas as pd
from pathlib import Path

Path("outputs").mkdir(parents=True, exist_ok=True)

rng = np.random.default_rng(3)
n = 500

anciennete_mois = rng.integers(1, 72, size=n)
appels_service_client = rng.poisson(1.5, size=n)
facture_mensuelle = rng.uniform(80, 600, size=n)
type_contrat = rng.choice(["Mensuel", "Annuel", "Deux ans"], size=n, p=[0.5, 0.3, 0.2])

# Probabilité de résiliation : augmente avec les appels au service client et
# les contrats mensuels, diminue avec l'ancienneté - une structure réaliste.
logit = (
    -2.0
    + 0.35 * appels_service_client
    - 0.03 * anciennete_mois
    + 0.004 * facture_mensuelle
    + pd.Series(type_contrat).map({"Mensuel": 1.2, "Annuel": -0.3, "Deux ans": -1.0}).to_numpy()
)
probabilite_churn = 1 / (1 + np.exp(-logit))
churn = rng.binomial(1, probabilite_churn)

clients = pd.DataFrame({
    "anciennete_mois": anciennete_mois,
    "appels_service_client": appels_service_client,
    "facture_mensuelle": facture_mensuelle.round(2),
    "type_contrat": type_contrat,
    "churn": churn,
})
print(clients["churn"].value_counts(normalize=True).round(3))

Comme souvent en pratique, les classes sont **déséquilibrées** : gardez ce chiffre en tête, il conditionne toute la suite (chapitre 2, §6, « le piège des classes déséquilibrées »).

## 2. Un pipeline de classification, de bout en bout

In [ ]:
from sklearn.compose import ColumnTransformer
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import train_test_split
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder, StandardScaler

X = clients.drop(columns="churn")
y = clients["churn"]

X_dev, X_test, y_dev, y_test = train_test_split(
    X, y, test_size=0.25, random_state=42, stratify=y,
)
X_train, X_val, y_train, y_val = train_test_split(
    X_dev, y_dev, test_size=0.2, random_state=42, stratify=y_dev,
)

print(f"Entraînement : {len(X_train)} observations")
print(f"Validation   : {len(X_val)} observations")
print(f"Test final   : {len(X_test)} observations (réservé jusqu'à la fin)")

colonnes_numeriques = ["anciennete_mois", "appels_service_client", "facture_mensuelle"]
colonnes_categorielles = ["type_contrat"]

pretraitement = ColumnTransformer([
    ("num", StandardScaler(), colonnes_numeriques),
    ("cat", OneHotEncoder(drop="first"), colonnes_categorielles),
])

pipeline_logit = Pipeline([
    ("pretraitement", pretraitement),
    ("modele", LogisticRegression()),
])

pipeline_logit.fit(X_train, y_train)
print("Accuracy validation :", round(pipeline_logit.score(X_val, y_val), 3))

## 3. Évaluer sur le jeu de validation

In [ ]:
from sklearn.metrics import (
    ConfusionMatrixDisplay, RocCurveDisplay, classification_report, f1_score,
    confusion_matrix, roc_auc_score,
)

y_pred_val = pipeline_logit.predict(X_val)
y_proba_val = pipeline_logit.predict_proba(X_val)[:, 1]   # probabilité de la classe "churn"

print(classification_report(y_val, y_pred_val, target_names=["reste", "churn"]))
print("ROC-AUC validation :", round(roc_auc_score(y_val, y_proba_val), 3))

In [ ]:
import matplotlib.pyplot as plt

fig, axes = plt.subplots(1, 2, figsize=(11, 4.5))

ConfusionMatrixDisplay(
    confusion_matrix(y_val, y_pred_val), display_labels=["reste", "churn"],
).plot(ax=axes[0], cmap="Blues")
axes[0].set_title("Matrice de confusion")

RocCurveDisplay.from_predictions(y_val, y_proba_val, ax=axes[1])
axes[1].set_title("Courbe ROC")

fig.tight_layout()
fig.savefig("outputs/evaluation_validation_churn.png", dpi=150)
plt.show()

**Lire la courbe ROC**

La courbe ROC (*Receiver Operating Characteristic*) trace le taux de vrais positifs contre le taux de faux positifs, pour **tous les seuils de décision possibles**. Un modèle sans pouvoir discriminant suit en moyenne la diagonale ; plus sa courbe s'en écarte vers le coin supérieur gauche, meilleur est son pouvoir de classement. L'aire sous la courbe (**AUC**) résume cette capacité dans un nombre compris entre 0 et 1 ; une valeur de 0,5 correspond au hasard et une valeur de 1 à un classement parfait.

## 4. Le seuil de décision n'est pas obligatoirement 0,5

Par défaut, `.predict()` classe en « churn » toute observation dont la probabilité prédite dépasse 0,5. Mais rien n'impose ce seuil : selon le coût métier d'un faux négatif (un client qui part sans qu'on ait rien tenté) par rapport à un faux positif (une offre de rétention envoyée à tort), on peut vouloir un seuil différent.

In [ ]:
from sklearn.metrics import precision_score, recall_score

for seuil in [0.3, 0.5, 0.7]:
    y_pred_seuil = (y_proba_val >= seuil).astype(int)
    precision = precision_score(y_val, y_pred_seuil, zero_division=0)
    rappel = recall_score(y_val, y_pred_seuil, zero_division=0)
    print(f"Seuil = {seuil} → précision = {precision:.3f}, rappel = {rappel:.3f}")

Abaisser le seuil **augmente le rappel** (on détecte plus de vrais churners) mais **diminue généralement la précision** (on se trompe aussi plus souvent sur des clients qui n'allaient pas partir). Ce compromis, illustré concrètement ici, est au cœur de toute décision de déploiement d'un classifieur.

## 5. Gérer le déséquilibre des classes

In [ ]:
pipeline_pondere = Pipeline([
    ("pretraitement", pretraitement),
    ("modele", LogisticRegression(class_weight="balanced")),
])
pipeline_pondere.fit(X_train, y_train)

y_pred_pondere = pipeline_pondere.predict(X_val)
print(classification_report(y_val, y_pred_pondere, target_names=["reste", "churn"]))

`class_weight="balanced"` donne automatiquement plus de poids à la classe minoritaire pendant l'entraînement, sans avoir à dupliquer ou supprimer des observations. Comparez le rappel de la classe « churn » sur la validation avec et sans cette option : c'est généralement là que l'écart est le plus visible.

## 6. Comparer plusieurs algorithmes, avec réglage d'hyperparamètres

Réutilisons `GridSearchCV`, déjà maîtrisé au Lab 3, pour comparer trois familles de modèles sur la même tâche.

In [ ]:
from sklearn.model_selection import GridSearchCV
from sklearn.neighbors import KNeighborsClassifier
from sklearn.tree import DecisionTreeClassifier

modeles_a_comparer = {
    "Régression logistique": (
        LogisticRegression(class_weight="balanced"),
        {"modele__C": [0.01, 0.1, 1, 10]},
    ),
    "KNN": (
        KNeighborsClassifier(),
        {"modele__n_neighbors": [3, 5, 9, 15, 25]},
    ),
    "Arbre de décision": (
        DecisionTreeClassifier(class_weight="balanced", random_state=42),
        {"modele__max_depth": [2, 3, 4, 5, 8, None]},
    ),
}

resultats = []
for nom, (estimateur, grille) in modeles_a_comparer.items():
    pipeline_candidat = Pipeline([("pretraitement", pretraitement), ("modele", estimateur)])
    recherche = GridSearchCV(
        pipeline_candidat,
        param_grid=grille,
        cv=5,
        scoring="f1",
        n_jobs=-1,
    )
    recherche.fit(X_train, y_train)
    score_validation = f1_score(
        y_val,
        recherche.best_estimator_.predict(X_val),
        zero_division=0,
    )
    resultats.append({
        "nom": nom,
        "recherche": recherche,
        "score_cv": recherche.best_score_,
        "score_validation": score_validation,
    })
    print(
        f"{nom:25s} → paramètres = {recherche.best_params_}, "
        f"F1 CV = {recherche.best_score_:.3f}, "
        f"F1 validation = {score_validation:.3f}"
    )

meilleur_resultat = max(resultats, key=lambda resultat: resultat["score_validation"])
print(f"\nModèle retenu : {meilleur_resultat['nom']}")

**Pourquoi `scoring="f1"` plutôt que l'accuracy ?**

Sur un jeu déséquilibré, optimiser l'accuracy par validation croisée pousserait naturellement `GridSearchCV` vers des modèles qui prédisent presque toujours la classe majoritaire (chapitre 2, §6). Le F1-score, qui combine précision et rappel, est un choix bien plus pertinent ici.

## 7. Évaluation finale sur le jeu de test

Le jeu de test n'a pas servi à choisir le modèle ni ses hyperparamètres. Une fois le modèle retenu sur la validation, on le réentraîne sur l'ensemble des données de développement, puis on évalue **une seule fois** ses performances sur le test.

In [ ]:
from sklearn.base import clone

modele_final = clone(meilleur_resultat["recherche"].best_estimator_)
modele_final.fit(X_dev, y_dev)

y_pred_test = modele_final.predict(X_test)
y_proba_test = modele_final.predict_proba(X_test)[:, 1]

print(classification_report(y_test, y_pred_test, target_names=["reste", "churn"]))
print("ROC-AUC test :", round(roc_auc_score(y_test, y_proba_test), 3))

matrice_test = confusion_matrix(y_test, y_pred_test)
fig, axes = plt.subplots(1, 2, figsize=(11, 4.5))
ConfusionMatrixDisplay(
    matrice_test, display_labels=["reste", "churn"],
).plot(ax=axes[0], cmap="Blues")
axes[0].set_title("Matrice de confusion - test final")
RocCurveDisplay.from_predictions(y_test, y_proba_test, ax=axes[1])
axes[1].set_title("Courbe ROC - test final")
fig.tight_layout()
fig.savefig("outputs/evaluation_finale_churn.png", dpi=150)
plt.show()

**Ne pas réutiliser le test**

Après cette évaluation, ne modifiez pas le modèle, ses hyperparamètres ou son seuil en fonction du score obtenu sur le test. Si vous le faites, ce jeu ne constitue plus une évaluation finale indépendante.

## 8. Travail 

**Jeu de données du projet : le Titanic, à nouveau**
Reprenez le jeu de données du Titanic préparé au Lab 2 (§9). Cette fois, l'objectif n'est plus seulement de le nettoyer, mais de réellement **prédire** la survie (`survived`) d'un passager.

In [ ]:
titanic = pd.read_csv(
    "https://raw.githubusercontent.com/mwaskom/seaborn-data/master/titanic.csv"
)

### Partie A - Préparer les jeux de données

1. Réappliquez (ou reconstruisez) le pipeline de prétraitement du Lab 2 sur les variables `pclass`, `sex`, `age`, `fare`, `embarked`, `sibsp`, `parch`, avec `survived` comme cible. Réservez d'abord 25 % des données au test final, puis séparez les données de développement en entraînement (80 %) et validation (20 %). Utilisez `stratify` à chaque séparation.
2. Le jeu est-il déséquilibré ? Donnez la proportion de survivants.

### Partie B - Premier modèle et métriques

3. Entraînez une `LogisticRegression` (pipeline complet). Calculez accuracy, précision, rappel, F1-score et ROC-AUC sur la validation.
4. Tracez la matrice de confusion et la courbe ROC pour la validation.
5. Une `DummyClassifier` (baseline) obtiendrait quelle accuracy sur la validation ? Votre modèle fait-il significativement mieux ?

### Partie C - Seuil et déséquilibre

6. Testez trois seuils de décision différents (par exemple 0.3, 0.5, 0.7) avec les probabilités prédites sur la validation. Lequel choisiriez-vous si l'objectif prioritaire était de **ne rater aucun survivant potentiel** (maximiser le rappel), quitte à avoir plus de fausses alertes ?
7. Comparez, avec et sans `class_weight="balanced"`, le rappel obtenu pour la classe minoritaire.

### Partie D - Comparaison de modèles et hyperparamètres

8. Comparez, par `GridSearchCV` avec `scoring="f1"`, une régression logistique (hyperparamètre `C`), un KNN (hyperparamètre `n_neighbors`) et un arbre de décision (hyperparamètre `max_depth`). Ajustez les modèles uniquement sur l'entraînement ; utilisez la validation pour retenir la famille de modèle.
9. Réentraînez le modèle retenu sur toutes les données de développement et évaluez-le une seule fois sur le test final. Quel F1-score obtient-il ? Ce résultat était-il celui que vous attendiez, compte tenu des données (variables catégorielles nombreuses, taille du jeu, etc.) ?
10. *(Bonus)* Affichez l'importance des variables pour votre meilleur modèle (`.coef_` pour la régression logistique, ou `.feature_importances_` pour l'arbre). Quelle variable semble la plus déterminante pour la survie ? Ce résultat correspond-il à ce que vous savez de l'histoire du naufrage ?

### Livrables

- un notebook ou script contenant le code des parties A à D ;
- les graphiques d'évaluation sur la validation et une évaluation finale du test dans `outputs/` ;
- un court paragraphe de conclusion, comparant les modèles testés et justifiant votre choix final.